In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/xndung/fidgety/data")
MODELS = ["vid", "imu", "mat", "fusion"]
FOLDS = [1]
RUNS = 1
EPOCHS = 5
SEED = 42

# Full experiment: FOLDS = list(range(1, 10)); RUNS = 20; EPOCHS = 1000


In [ ]:
import subprocess
import sys

SOURCE_DIR = Path("/kaggle/working/Fidgety")
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
if SOURCE_DIR.exists():
    subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(SOURCE_DIR / "requirements.txt")], check=True)
REVISION = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source:", REVISION)


In [ ]:
import json
import os
from datetime import datetime, timezone
import torch

SCRIPTS = {
    "vid": "vitpose_kpc_vel_conv_best_model_9fold.py",
    "imu": "imu_acc_gyr_conv_best_model_9fold.py",
    "mat": "mat_conv_best_model_9fold.py",
    "fusion": "mat_vitpose_imu_conv_best_model_9fold.py",
}
assert MODELS and len(set(MODELS)) == len(MODELS) and all(m in SCRIPTS for m in MODELS)
assert FOLDS and len(set(FOLDS)) == len(FOLDS) and all(isinstance(f, int) and 1 <= f <= 9 for f in FOLDS)
assert isinstance(RUNS, int) and RUNS > 0 and isinstance(EPOCHS, int) and EPOCHS > 0
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Settings before training.")
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(DATA_ROOT)
if not (DATA_ROOT / "data_mat_9fold").is_dir() and (DATA_ROOT / "data" / "data_mat_9fold").is_dir():
    DATA_ROOT = DATA_ROOT / "data"

required = set()
for model in MODELS:
    modalities = ["mat", "imu", "vitpose_kpc"] if model == "fusion" else [{"vid": "vitpose_kpc", "imu": "imu", "mat": "mat"}[model]]
    for fold in FOLDS:
        required.add(f"random_indices_9fold/random_indices_train_9fold_fold_{fold}.txt")
        for modality in modalities:
            prefix = "vitpose" if modality == "vitpose_kpc" else modality
            for split in ["train", "test"]:
                for kind in ["input", "output"]:
                    required.add(f"data_{modality}_9fold/{kind}_{prefix}_9fold_fold_{fold}_{split}.txt")
missing = [str(DATA_ROOT / name) for name in sorted(required) if not (DATA_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(missing))

OUTPUT_DIR = Path("/kaggle/working/outputs") / datetime.now(timezone.utc).strftime("all_models_%Y%m%d_%H%M%S_%f")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
CSV_PATH = OUTPUT_DIR / "results.csv"
CONFIG = dict(data_root=str(DATA_ROOT), models=MODELS, folds=FOLDS, runs=RUNS,
              epochs=EPOCHS, seed=SEED, source_commit=REVISION)
(OUTPUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print("GPU:", torch.cuda.get_device_name(0))
print("Data:", DATA_ROOT)
print("Jobs:", len(MODELS) * len(FOLDS), "| Runs per job:", RUNS, "| Max epochs:", EPOCHS)
print("CSV:", CSV_PATH)


In [ ]:
import csv
import time

if CSV_PATH.exists():
    raise FileExistsError("Results already exist. Rerun the configuration cell to start a new experiment.")

for model in MODELS:
    for fold in FOLDS:
        job_dir = OUTPUT_DIR / model / f"fold_{fold}"
        job_dir.mkdir(parents=True, exist_ok=False)
        env = os.environ.copy()
        env.pop("FIDGETY_DATA_ZIP", None)
        env.update(FIDGETY_DATA_ROOT=str(DATA_ROOT), FIDGETY_FOLDS=str(fold),
                   FIDGETY_RUNS=str(RUNS), FIDGETY_EPOCHS=str(EPOCHS),
                   FIDGETY_MODEL=model, FIDGETY_RESULTS_CSV=str(CSV_PATH),
                   FIDGETY_SEED=str(SEED), PYTHONUNBUFFERED="1")
        print(f"\n{'=' * 16} {model.upper()} | FOLD {fold} {'=' * 16}", flush=True)
        started = time.monotonic()
        try:
            with (job_dir / "train.log").open("w", encoding="utf-8") as log:
                with subprocess.Popen([sys.executable, "-u", str(SOURCE_DIR / SCRIPTS[model])],
                                      cwd=job_dir, env=env, stdout=subprocess.PIPE,
                                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
                    try:
                        for line in process.stdout:
                            print(line, end="", flush=True)
                            log.write(line)
                            log.flush()
                        returncode = process.wait()
                    except BaseException:
                        process.terminate()
                        process.wait()
                        raise
            if returncode:
                raise RuntimeError(f"{model}, fold {fold} failed; see {job_dir / 'train.log'}")
            if not CSV_PATH.exists():
                raise RuntimeError("No metrics CSV produced; check the source version.")
            with CSV_PATH.open(newline="", encoding="utf-8") as stream:
                rows = [row for row in csv.DictReader(stream) if row["model"] == model and int(row["fold"]) == fold]
                if len(rows) != RUNS:
                    raise RuntimeError("Unexpected number of completed runs.")
            print(f"Completed in {time.monotonic() - started:.1f} seconds", flush=True)
        finally:
            if CSV_PATH.exists():
                print("Results saved:", CSV_PATH, flush=True)
print("Saved:", CSV_PATH)


In [ ]:
import pandas as pd
from IPython.display import FileLink, display

results = pd.read_csv(CSV_PATH)
expected_rows = len(MODELS) * len(FOLDS) * RUNS
if len(results) != expected_rows:
    print(f"Partial results: {len(results)}/{expected_rows} runs")
display(results)
summary = results.groupby("model", sort=False).agg(
    completed_runs=("run", "count"),
    test_accuracy=("test_accuracy", "mean"),
    sensitivity=("sensitivity", "mean"),
    specificity=("specificity", "mean"),
    balanced_accuracy=("balanced_accuracy", "mean"),
)
display(summary)
print("Accuracy/sensitivity/specificity/balanced_accuracy use sigmoid probabilities, threshold 0.5.")
print("Columns ending in _logits retain the author's training metric (threshold 0.5 on raw logits).")
print("Summary values are unweighted means across the completed fold/run rows.")
print("CSV:", CSV_PATH)
display(FileLink(str(CSV_PATH)))
